[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_FaceStyle.ipynb)


# 顔スタイル変換デモ（Face Style Transfer）

顔写真を **アニメキャラ風** や **絵画・映画イラスト風** に変換するデモです．  
[AnimeGANv2](https://github.com/bryandlee/animegan2-pytorch)（`torch.hub`）を Colab の GPU 上で動かします．

**実行環境**: Google Colab（ランタイム → GPU: **T4** 推奨）

## セルの進め方
1. **設定**（Webカメラの左右反転・変換サイズなど）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル画像のダウンロード**
4. **Gradio の起動**

> 初回はモデルのダウンロードに数十秒〜数分かかることがあります．  
> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから設定セルとセル3以降を再実行してください．


## 0. 設定

- カメラ映像が左右反転して見える場合は，次のセルの `MIRROR_WEBCAM` を切り替えてください（`True` = ミラー，`False` = 反転なし）．
- `PAINT_SIZE` は変換解像度です．512 が品質と速度のバランスが良いです．
- 変更後は **Gradio 起動セル**を再実行してください．


In [ ]:
# Webカメラの左右反転（ミラー表示）
# True  : 左右反転する（Gradio のデフォルトに近い自撮り表示）
# False : 左右反転しない
MIRROR_WEBCAM = True

# AnimeGANv2 の入力解像度（正方形）
PAINT_SIZE = 512

# 顔検出後に周囲へ余白を足す倍率（1.0 = 検出枠そのもの）
FACE_MARGIN = 0.45

# 既定のスタイルキー（Gradio の初期選択）
#   face_paint_512_v2 / face_paint_512_v1 / celeba_distill / paprika
DEFAULT_STYLE = "face_paint_512_v2"

print(f"MIRROR_WEBCAM = {MIRROR_WEBCAM}")
print(f"PAINT_SIZE = {PAINT_SIZE}")
print(f"FACE_MARGIN = {FACE_MARGIN}")
print(f"DEFAULT_STYLE = {DEFAULT_STYLE}")


## 1. ライブラリのインストール


In [ ]:
# Colab 標準の torch / torchvision / gradio / opencv / Pillow を利用
# AnimeGANv2 は torch.hub 経由で取得する
!pip install -q -U "tqdm"


## 2. ライブラリの読み込み，変数のインスタンス化

サンプル顔写真（日本人・アジア系を含む）をインターネットからダウンロードし，AnimeGANv2 の各スタイルモデルを準備します．  
初回は重みのダウンロードに時間がかかることがあります．


In [ ]:
from __future__ import annotations

import urllib.request
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from torchvision.transforms.functional import to_pil_image, to_tensor

# ------------------------------------------------------------
# 定数・サンプル画像 URL・スタイル定義
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_face_style")
FONT_DIR = Path("fonts")
FONT_PATH = FONT_DIR / "NotoSansJP-VF.ttf"
FONT_URL = (
    "https://raw.githubusercontent.com/googlefonts/noto-cjk/main/"
    "Sans/Variable/TTF/Subset/NotoSansJP-VF.ttf"
)

HUB_REPO = "bryandlee/animegan2-pytorch:main"

# スタイルキー → (日本語ラベル, 短い説明)
STYLE_INFO: dict[str, tuple[str, str]] = {
    "face_paint_512_v2": (
        "アニメキャラ風（推奨）",
        "顔写真をアニメのキャラクター風に変換します",
    ),
    "face_paint_512_v1": (
        "アニメ似顔絵風",
        "やや綺麗めのアニメ似顔絵タッチです",
    ),
    "celeba_distill": (
        "やさしいイラスト風",
        "柔らかい線のイラスト寄りの変換です",
    ),
    "paprika": (
        "映画イラスト・絵画風",
        "映画『パプリカ』風の絵画寄りのタッチです",
    ),
}

# 公開画像（Wikimedia / Pexels）．日本人・アジア系を含む．
SAMPLE_IMAGE_SOURCES: list[tuple[str, str, str]] = [
    (
        "happy_japanese_woman.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/b/b0/Smiling_Japanese_Woman.jpg",
        "笑顔（日本人）",
    ),
    (
        "happy_japanese_smile.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/d/d1/Smiling_Ai_Hongo_%282024%2902.jpg",
        "スマイル（日本人）",
    ),
    (
        "neutral_japanese.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/9/90/Geisha_face_%285025641801%29.jpg",
        "ポートレート（日本人）",
    ),
    (
        "asian_portrait.jpg",
        "https://images.pexels.com/photos/1239291/pexels-photo-1239291.jpeg?auto=compress&cs=tinysrgb&w=640",
        "ポートレート（アジア系）",
    ),
    (
        "serious.jpg",
        "https://images.pexels.com/photos/2379004/pexels-photo-2379004.jpeg?auto=compress&cs=tinysrgb&w=640",
        "真剣な表情",
    ),
]

USER_AGENT = (
    "Mozilla/5.0 (compatible; OpenCampusDemo/1.0; "
    "+https://github.com/yryo1005/OpenCampus_Demo)"
)


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（時間がかかります）．")
    return "cpu"


def download_file(url: str, save_path: Path, max_side: int = 1280) -> Path:
    """URL から画像をダウンロードし，必要なら長辺を縮小して保存する．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス
        max_side (int): 長辺の上限ピクセル（既定 1280）

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        raw = response.read()
    arr = np.frombuffer(raw, dtype=np.uint8)
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"画像のデコードに失敗しました: {url}")
    h, w = bgr.shape[:2]
    long_side = max(h, w)
    if long_side > max_side:
        scale = max_side / float(long_side)
        bgr = cv2.resize(
            bgr,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    ok, encoded = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError(f"画像のエンコードに失敗しました: {save_path}")
    save_path.write_bytes(encoded.tobytes())
    return save_path


def download_font(url: str, save_path: Path) -> Path:
    """日本語表示用フォントをダウンロードする（既存ならスキップ）．

    Args:
        url (str): フォントの URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したフォントのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=120) as response:
        save_path.write_bytes(response.read())
    return save_path


def prepare_sample_images(
    sources: list[tuple[str, str, str]],
    sample_dir: Path,
) -> list[tuple[str, Path]]:
    """サンプル顔写真をダウンロードし，ラベルとパスの一覧を返す．

    Args:
        sources (list[tuple[str, str, str]]): (ファイル名, URL, 表示ラベル) のリスト
        sample_dir (Path): 保存先ディレクトリ

    Returns:
        list[tuple[str, Path]]: (表示ラベル, ローカルパス) のリスト
    """
    prepared: list[tuple[str, Path]] = []
    for filename, url, label in tqdm(sources, desc="サンプル画像DL", leave=False):
        path = download_file(url, sample_dir / filename)
        print(f"  {label}: {path} ({path.stat().st_size} bytes)")
        prepared.append((label, path))
    return prepared


def load_face_cascade() -> cv2.CascadeClassifier:
    """OpenCV の顔検出用 Haar Cascade を読み込む．

    Returns:
        cv2.CascadeClassifier: 顔検出器

    Raises:
        RuntimeError: cascade ファイルが読めない場合
    """
    cascade_path = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
    cascade = cv2.CascadeClassifier(str(cascade_path))
    if cascade.empty():
        raise RuntimeError(f"顔検出 cascade を読めません: {cascade_path}")
    return cascade


def load_style_models(
    style_keys: list[str],
    device: str,
) -> dict[str, torch.nn.Module]:
    """AnimeGANv2 のスタイル別 Generator を torch.hub から読み込む．

    Args:
        style_keys (list[str]): 事前学習スタイル名のリスト
        device (str): "cuda" または "cpu"

    Returns:
        dict[str, torch.nn.Module]: スタイル名 → eval 済み Generator
    """
    models: dict[str, torch.nn.Module] = {}
    for key in tqdm(style_keys, desc="スタイルモデル読込", leave=False):
        print(f"読み込み中: {key}")
        model = torch.hub.load(
            HUB_REPO,
            "generator",
            pretrained=key,
            device=device,
            progress=True,
            trust_repo=True,
        )
        model.eval()
        models[key] = model
    return models


def to_rgb_uint8(image) -> np.ndarray | None:
    """Gradio / PIL / ndarray 入力を RGB uint8 (H, W, 3) に揃える．

    Args:
        image: Gradio Image の入力（None / PIL.Image / np.ndarray）

    Returns:
        np.ndarray | None: RGB 画像．入力が無い場合は None
    """
    if image is None:
        return None
    if isinstance(image, Image.Image):
        return np.asarray(image.convert("RGB"))
    arr = np.asarray(image)
    if arr.ndim == 2:
        return cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    if arr.shape[2] == 4:
        return arr[:, :, :3].astype(np.uint8)
    return arr.astype(np.uint8)


def detect_largest_face(
    rgb: np.ndarray,
    cascade: cv2.CascadeClassifier,
    margin: float = 0.45,
) -> tuple[int, int, int, int] | None:
    """画像から最も大きい顔のバウンディングボックスを返す．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        cascade (cv2.CascadeClassifier): 顔検出器
        margin (float): 検出枠に対する余白倍率

    Returns:
        tuple[int, int, int, int] | None: (x1, y1, x2, y2)．未検出時は None
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(48, 48),
    )
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad_x = int(w * margin)
    pad_y = int(h * margin)
    h_img, w_img = rgb.shape[:2]
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(w_img, x + w + pad_x)
    y2 = min(h_img, y + h + pad_y)
    return x1, y1, x2, y2


def crop_square_face(
    rgb: np.ndarray,
    box: tuple[int, int, int, int] | None,
) -> tuple[np.ndarray, str]:
    """顔枠があれば顔中心の正方形を切り出し，無ければ画像中央を切り出す．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        box (tuple[int, int, int, int] | None): 顔の (x1, y1, x2, y2)

    Returns:
        tuple[np.ndarray, str]: (正方形 RGB 画像, 切り出し方法の説明)
    """
    h, w = rgb.shape[:2]
    if box is not None:
        x1, y1, x2, y2 = box
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        side = max(x2 - x1, y2 - y1)
        note = "顔を検出して切り出しました"
    else:
        cx, cy = w // 2, h // 2
        side = min(h, w)
        note = "顔を検出できなかったため，画像の中央を切り出しました"

    half = side // 2
    x1 = max(0, cx - half)
    y1 = max(0, cy - half)
    x2 = min(w, x1 + side)
    y2 = min(h, y1 + side)
    # 端に寄った場合の補正
    x1 = max(0, x2 - side)
    y1 = max(0, y2 - side)
    crop = rgb[y1:y2, x1:x2]
    return crop, note


def stylize_face(
    face_rgb: np.ndarray,
    model: torch.nn.Module,
    device: str,
    size: int,
) -> Image.Image:
    """顔画像を AnimeGANv2 でスタイル変換する．

    Args:
        face_rgb (np.ndarray): RGB 顔画像，形状 (H, W, 3)
        model (torch.nn.Module): AnimeGANv2 Generator
        device (str): "cuda" または "cpu"
        size (int): 変換解像度（正方形）

    Returns:
        PIL.Image.Image: スタイル変換後の RGB 画像（size × size）
    """
    pil = Image.fromarray(face_rgb).convert("RGB")
    pil = pil.resize((size, size), Image.LANCZOS)
    with torch.no_grad():
        x = to_tensor(pil).unsqueeze(0) * 2.0 - 1.0
        y = model(x.to(device)).cpu()[0]
        y = (y * 0.5 + 0.5).clamp(0.0, 1.0)
    return to_pil_image(y)


def make_side_by_side(
    before: Image.Image,
    after: Image.Image,
    style_label: str,
) -> np.ndarray:
    """変換前後を横並びにし，ラベルを付けた RGB 画像を返す．

    Args:
        before (PIL.Image.Image): 変換前
        after (PIL.Image.Image): 変換後
        style_label (str): 変換後側に書くスタイル名

    Returns:
        np.ndarray: 横並び RGB 画像，形状 (H, 2W, 3)
    """
    size = before.size[0]
    after = after.resize((size, size), Image.LANCZOS)
    canvas = Image.new("RGB", (size * 2, size))
    canvas.paste(before, (0, 0))
    canvas.paste(after, (size, 0))

    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype(str(FONT_PATH), 40)
    except OSError:
        font = ImageFont.load_default()
    bar_h = 56
    draw.rectangle([(0, 0), (size * 2, bar_h)], fill=(20, 20, 20))
    draw.text((12, 10), "変換前", fill=(255, 255, 255), font=font)
    draw.text((size + 12, 10), f"変換後（{style_label}）", fill=(255, 220, 120), font=font)
    return np.asarray(canvas)


def style_key_from_label(label: str) -> str:
    """Gradio の日本語ラベルからスタイルキーを求める．

    Args:
        label (str): ドロップダウン表示名

    Returns:
        str: STYLE_INFO のキー
    """
    for key, (ja, _) in STYLE_INFO.items():
        if ja == label:
            return key
    return DEFAULT_STYLE


def convert_face_style(
    image,
    style_label: str,
    mirror: bool = False,
) -> tuple[np.ndarray | None, np.ndarray | None, str]:
    """顔写真を選択スタイルへ変換する（Gradio コールバック）．

    Args:
        image: Gradio Image 入力（カメラ／アップロード／サンプル）
        style_label (str): スタイルの日本語名
        mirror (bool): True のとき左右反転してから処理する

    Returns:
        tuple[np.ndarray | None, np.ndarray | None, str]:
            (スタイル変換画像, 横並び比較画像, 説明テキスト)
    """
    rgb = to_rgb_uint8(image)
    if rgb is None:
        msg = (
            "画像がありません．カメラ撮影・アップロード・サンプルのいずれかを選んでください．"
        )
        return None, None, msg

    if mirror:
        rgb = np.ascontiguousarray(rgb[:, ::-1, :])

    style_key = style_key_from_label(style_label)
    style_ja, style_desc = STYLE_INFO[style_key]
    model = style_models[style_key]

    with tqdm(total=3, desc="スタイル変換", leave=False) as pbar:
        box = detect_largest_face(rgb, face_cascade, margin=FACE_MARGIN)
        face_rgb, crop_note = crop_square_face(rgb, box)
        pbar.update(1)

        before = Image.fromarray(face_rgb).resize(
            (PAINT_SIZE, PAINT_SIZE), Image.LANCZOS
        )
        after = stylize_face(face_rgb, model, device, PAINT_SIZE)
        pbar.update(1)

        compare = make_side_by_side(before, after, style_ja)
        pbar.update(1)

    text = (
        f"【スタイル】{style_ja}\n"
        f"{style_desc}\n\n"
        f"【切り出し】{crop_note}\n"
        f"【解像度】{PAINT_SIZE}×{PAINT_SIZE}\n"
        f"【モデル】AnimeGANv2 / {style_key}"
    )
    return np.asarray(after), compare, text


def build_demo(
    sample_items: list[tuple[str, Path]],
    mirror_webcam: bool,
) -> gr.Blocks:
    """Gradio UI を構築する．

    Args:
        sample_items (list[tuple[str, Path]]): (表示ラベル, 画像パス)
        mirror_webcam (bool): カメラ入力を左右反転するか

    Returns:
        gr.Blocks: Gradio デモ
    """
    example_paths = [str(path) for _, path in sample_items]
    style_choices = [STYLE_INFO[k][0] for k in STYLE_INFO]
    default_label = STYLE_INFO.get(DEFAULT_STYLE, STYLE_INFO["face_paint_512_v2"])[0]

    with gr.Blocks(title="顔スタイル変換デモ") as demo:
        with gr.Row():
            with gr.Column():
                image_in = gr.Image(
                    label="顔写真（カメラ / アップロード）",
                    type="numpy",
                    sources=["webcam", "upload"],
                    webcam_options=gr.WebcamOptions(mirror=mirror_webcam),
                )
                style_dd = gr.Dropdown(
                    choices=style_choices,
                    value=default_label,
                    label="スタイル",
                    info="アニメキャラ化／絵画寄りのタッチを選べます",
                )
                mirror_flag = gr.Checkbox(
                    label="入力画像を左右反転して変換する",
                    value=False,
                    info="アップロード画像の向きが逆のときだけオンにしてください（カメラは上のミラー設定を利用）",
                )
                run_btn = gr.Button("スタイル変換", variant="primary")
            with gr.Column():
                image_out = gr.Image(label="変換結果", type="numpy")
                compare_out = gr.Image(label="変換前 / 変換後", type="numpy")
                text_out = gr.Textbox(label="説明", lines=8)

        gr.Examples(
            examples=example_paths,
            inputs=[image_in],
            label="サンプル画像（クリックで入力）",
            examples_per_page=8,
        )

        run_btn.click(
            fn=convert_face_style,
            inputs=[image_in, style_dd, mirror_flag],
            outputs=[image_out, compare_out, text_out],
        )
        image_in.change(
            fn=convert_face_style,
            inputs=[image_in, style_dd, mirror_flag],
            outputs=[image_out, compare_out, text_out],
        )
        style_dd.change(
            fn=convert_face_style,
            inputs=[image_in, style_dd, mirror_flag],
            outputs=[image_out, compare_out, text_out],
        )
    return demo


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
device = resolve_device()
download_font(FONT_URL, FONT_PATH)
print(f"フォント: {FONT_PATH} ({FONT_PATH.stat().st_size} bytes)")
sample_items = prepare_sample_images(SAMPLE_IMAGE_SOURCES, SAMPLE_DIR)
face_cascade = load_face_cascade()
style_models = load_style_models(list(STYLE_INFO.keys()), device)
print("初期化完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行

UI が起動したら，サンプル画像をクリックするか，カメラで顔を撮影して「スタイル変換」を押してください．


In [ ]:
demo = build_demo(sample_items, mirror_webcam=MIRROR_WEBCAM)
demo.launch(share=True, debug=False)
